In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import mplhep as hep
import numpy as np
import uproot

plt.style.use(hep.style.CMS)
hep.style.use("CMS")
formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((-3, 3))
plt.rcParams.update({"font.size": 20})

In [2]:
import HH4b

MAIN_DIR = Path(HH4b.__file__).parents[2]

passbin = "passbin1"

plot_dir = MAIN_DIR / f"plots/FTests/run3-bdt-26Jan15/{passbin}"
plot_dir.mkdir(exist_ok=True, parents=True)

In [3]:
def p_value(data_ts: float, toy_ts: list[float]):
    return np.mean(toy_ts >= data_ts)


def F_statistic(
    ts_low: list[float],
    ts_high: list[float],
    ord_low: int,
    ord_high: int,
    num_bins: int = 10 * 14,
    dim: int = 2,
):
    numerator = (ts_low - ts_high) / (ord_high - ord_low)
    denominator = ts_high / (num_bins - (ord_high + dim))

    return numerator / denominator

In [5]:
local_cards_dir = MAIN_DIR / "cards/f_tests/run3-bdt-26Jan15"
print(local_cards_dir)
test_orders = [0]
test_statistics = {}

for o1 in test_orders:
    tdict = {"toys": {}, "data": {}, "ftoys": {}, "fdata": {}}
    tlabel = f"{o1}"

    for nTF in [o1, o1 + 1]:
        tflabel = f"{nTF}"

        # test statistics for toys generated by (o1, o2) order model
        file = uproot.concatenate(
            f"{local_cards_dir}/{passbin}_nTF_{nTF}/higgsCombineToys{tlabel}.GoodnessOfFit.mH125.*.root"
        )
        tdict["toys"][tflabel] = np.array(file["limit"])

        # data test statistic
        file = uproot.concatenate(
            f"{local_cards_dir}/{passbin}_nTF_{nTF}/higgsCombineData.GoodnessOfFit.mH125.root"
        )
        tdict["data"][tflabel] = file["limit"][0]

        if nTF != o1:
            tdict["ftoys"][tflabel] = F_statistic(
                tdict["toys"][tlabel], tdict["toys"][tflabel], o1, nTF
            )
            tdict["fdata"][tflabel] = F_statistic(
                tdict["data"][tlabel], tdict["data"][tflabel], o1, nTF
            )

    test_statistics[tlabel] = tdict

/home/users/dprimosc/HH4b/cards/f_tests/run3-bdt-26Jan15


ValueError: no TTrees found
in file /home/users/dprimosc/HH4b/cards/f_tests/run3-bdt-26Jan15/passbin1_nTF_0/higgsCombineToys0.GoodnessOfFit.mH125.444.root

In [ ]:
from scipy import stats

category_label = {
    "passbin1": "ggF Category 1",
    "passbin2": "ggF Category 2",
    "passbin3": "ggF Category 3",
    "passvbf": "VBF Category",
    "passbin0": "Combined",
}


def plot_tests(
    data_ts: float,
    toy_ts: np.ndarray,
    name: str,
    title: str = None,
    bins: int = 15,
    fit: str = None,
    fdof2: int = None,
    xlim: float = None,
):
    plot_max = max(np.max(toy_ts), data_ts)
    # plot_max = max(np.max(toy_ts), data_ts) if fit != "chi2" else 200
    # plot_min = min(np.min(toy_ts), data_ts, 0)
    plot_min = 0
    pval = p_value(data_ts, toy_ts)

    plt.figure(figsize=(12, 8))
    h = plt.hist(
        toy_ts,
        np.linspace(plot_min, plot_max if xlim is None else xlim, bins + 1),
        color="#8C8C8C",
        histtype="step",
        label=f"{len(toy_ts)} Toys",
    )
    plt.axvline(data_ts, color="#FF502E", linestyle=":", label=rf"Data ($p$-value = {pval:.2f})")

    if fit is not None:
        x = np.linspace(plot_min + 0.01, plot_max, 100)

        if fit == "chi2":
            res = stats.fit(stats.chi2, toy_ts, [(0, 200)])
            pdf = stats.chi2.pdf(x, res.params.df)
            label = rf"$\chi^2_{{DoF = {res.params.df:.2f}}}$ Fit"
        elif fit == "f":
            pdf = stats.f.pdf(x, 1, fdof2)
            label = rf"$F-dist_{{DoF = (1, {fdof2})}}$"
        else:
            raise ValueError("Invalid fit")

        plt.plot(
            x,
            pdf * (np.max(h[0]) / np.max(pdf)),
            color="#1f78b4",
            linestyle="--",
            # alpha=0.6,
            label=label,
        )

    hep.cms.label(
        "Work in Progress",
        data=True,
        lumi=61,
        year=None,
    )

    _ = plt.legend(title=category_label[passbin])
    plt.title(title)
    plt.ylabel("Number of Toys")
    plt.xlabel("Test Statistic")

    plt.savefig(f"{plot_dir}/{name}.pdf", bbox_inches="tight")

In [ ]:
o1 = 0  # order being tested
tlabel = f"{o1}"

data_ts, toy_ts = test_statistics[tlabel]["data"][tlabel], test_statistics[tlabel]["toys"][tlabel]
plot_tests(data_ts, toy_ts, "gof" + tlabel, fit="chi2", title=f"GOF test, order {o1}", bins=20)

ord1 = o1 + 1
tflabel = f"{ord1}"
data_ts, toy_ts = pval = (
    test_statistics[tlabel]["fdata"][tflabel],
    test_statistics[tlabel]["ftoys"][tflabel],
)
plot_tests(
    data_ts, toy_ts, f"f{tlabel}_{tflabel}", title=f"F-test, order {o1} vs. {ord1}", xlim=100
)

In [ ]:
passbin = "passbin0"
local_cards_dir = MAIN_DIR / "src/HH4b/cards/run3-bdt-february10-glopartv2-bdtv13-ntf0000-all-sig"
plot_dir = MAIN_DIR / "plots/run3-bdt-february10-glopartv2-bdtv13-ntf0000"
plot_dir.mkdir(exist_ok=True, parents=True)

tdict = {}

# test statistics for toys
file = uproot.concatenate(f"{local_cards_dir}/higgsCombineToys.GoodnessOfFit.mH125.*.root")
quantileExpected = np.array(file["quantileExpected"])
tdict["toys"] = np.array(file["limit"])[quantileExpected > -2]


# data test statistic
file = uproot.concatenate(f"{local_cards_dir}/higgsCombineData.GoodnessOfFit.mH125.root")
tdict["data"] = file["limit"][0]


data_ts, toy_ts = tdict["data"], tdict["toys"]
plot_tests(data_ts, toy_ts, "gof_combined", fit="chi2", title="GOF test, combined", bins=20)